##Install Requirement

In [ ]:
!nvidia-smi

In [ ]:
!git clone https://github.com/alexandrachirita98/MedViT-Quantum/

In [ ]:
%cd /kaggle/working/MedViT-Quantum

In [ ]:
%pwd

In [ ]:
pip install -r requirements.txt

## IBM Quantum: install + credentials

In [ ]:
# Pennylane's Qiskit plugin + IBM Runtime client.
!pip install -U pennylane-qiskit qiskit-ibm-runtime

### How to get an IBM Quantum API token

1. Create an account at https://quantum.ibm.com (free).
2. On the dashboard, copy your **API token** from the top-right corner.
3. Add the token as a Kaggle secret named `IBM_QUANTUM_TOKEN`:
   *Notebook settings → Add-ons → Secrets → Add a new secret.*
4. Make sure **Internet** is enabled in the notebook settings (Add-ons → Internet).

On Colab use `from google.colab import userdata; userdata.get('IBM_QUANTUM_TOKEN')`.
Locally: `export IBM_QUANTUM_TOKEN=...` before launching Jupyter.

**Free-tier limits** (2026): ~10 min/month of QPU time on most
Heron/Eagle backends. Plan to send only a handful of inference
samples — full test-set eval is not affordable on the free tier.

In [ ]:
import os

IBM_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    IBM_TOKEN = UserSecretsClient().get_secret('IBM_QUANTUM_TOKEN')
    print('Loaded IBM token from Kaggle secret.')
except Exception:
    IBM_TOKEN = os.environ.get('IBM_QUANTUM_TOKEN')
    if IBM_TOKEN:
        print('Loaded IBM token from environment.')

if not IBM_TOKEN:
    raise RuntimeError(
        'IBM_QUANTUM_TOKEN is not set. Add it as a Kaggle secret '
        '(or env var) before continuing.'
    )

from qiskit_ibm_runtime import QiskitRuntimeService
QiskitRuntimeService.save_account(
    token=IBM_TOKEN,
    channel='ibm_quantum',
    overwrite=True,
    set_as_default=True,
)

service = QiskitRuntimeService()
print('Available backends:')
for b in service.backends(operational=True, simulator=False):
    print(' -', b.name, '|', b.num_qubits, 'qubits |',
          b.status().pending_jobs, 'pending jobs')

In [ ]:
print("PyTorch", torch.__version__)
print("Torchvision", torchvision.__version__)
print("Torchattacks", torchattacks.__version__)
print("Numpy", np.__version__)
print("Medmnist", medmnist.__version__)

##Dataset

data_flag =  
[tissuemnist, pathmnist, chestmnist, dermamnist, octmnist, pnemoniamnist, retinamnist, breastmnist, bloodmnist, tissuemnist, organamnist, organcmnist, organsmnist]

In [ ]:
data_flag = 'retinamnist'
# [tissuemnist, pathmnist, chestmnist, dermamnist, octmnist,
# pnemoniamnist, retinamnist, breastmnist, bloodmnist, tissuemnist, organamnist, organcmnist, organsmnist]
download = True

NUM_EPOCHS = 10
BATCH_SIZE = 10
lr = 0.005

info = INFO[data_flag]
task = info['task']
n_channels = info['n_channels']
n_classes = len(info['label'])

DataClass = getattr(medmnist, info['python_class'])

print("number of channels : ", n_channels)
print("number of classes : ", n_classes)

In [ ]:
from torchvision.transforms.transforms import Resize
# preprocessing
train_transform = transforms.Compose([
    transforms.Resize(224),
    transforms.Lambda(lambda image: image.convert('RGB')),
    torchvision.transforms.AugMix(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[.5], std=[.5])
])
test_transform = transforms.Compose([
    transforms.Resize(224),
    transforms.Lambda(lambda image: image.convert('RGB')),
    transforms.ToTensor(),
    transforms.Normalize(mean=[.5], std=[.5])
])

# load the data
train_dataset = DataClass(split='train', transform=train_transform, download=download)
test_dataset = DataClass(split='test', transform=test_transform, download=download)

# pil_dataset = DataClass(split='train', download=download)

# encapsulate data into dataloader form
train_loader = data.DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True)
train_loader_at_eval = data.DataLoader(dataset=train_dataset, batch_size=2*BATCH_SIZE, shuffle=False)
test_loader = data.DataLoader(dataset=test_dataset, batch_size=2*BATCH_SIZE, shuffle=False)

In [ ]:
print(train_dataset)
print("===================")
print(test_dataset)

##Model

MedViTs ---> QMedViT_Softmax_Only

In [ ]:
from quantum_variants.softmax_only import QMedViT_Softmax_Only

# Train on the analytic simulator (qpu_mode=False) — training on real
# hardware is not feasible (cost + time). We'll load these weights
# into a hardware-backed model for inference further down.
model = QMedViT_Softmax_Only(
    stem_chs=[64, 32, 64], depths=[3, 4, 10, 3], path_dropout=0.1,
    num_classes=n_classes,
    qpu_mode=False,
).cuda()

## Train

In [ ]:
# define loss function and optimizer
if task == "multi-label, binary-class":
    criterion = nn.BCEWithLogitsLoss()
else:
    criterion = nn.CrossEntropyLoss()

optimizer = optim.SGD(model.parameters(), lr=lr, momentum=0.9)

In [ ]:
# train

for epoch in range(NUM_EPOCHS):
    train_correct = 0
    train_total = 0
    test_correct = 0
    test_total = 0
    print('Epoch [%d/%d]'% (epoch+1, NUM_EPOCHS))
    model.train()
    for inputs, targets in tqdm(train_loader):
        inputs, targets = inputs.cuda(), targets.cuda()
        # forward + backward + optimize
        optimizer.zero_grad()
        outputs = model(inputs)

        if task == 'multi-label, binary-class':
            targets = targets.to(torch.float32)
            loss = criterion(outputs, targets)
        else:
            targets = targets.squeeze().long()
            loss = criterion(outputs, targets)

        loss.backward()
        optimizer.step()

## Inference on real IBM Quantum hardware

**Cost reality check.** A single forward pass with the default config
issues `B*H*N` ≈ 11,760 circuit calls **per quantum LTB** (and we have
three quantum LTBs in stage 3). At ~1s/call on an IBM backend that is
many hours per single image, well beyond the free tier. To get a
tractable demo, we both:

- Quantize only the last LTB of stage 3 (`quantum_block_indices=(2,)`).
- Use `BATCH_SIZE=1` and run on a handful of test images.

Adjust `IBM_BACKEND` below to whatever your account exposes.

In [ ]:
IBM_BACKEND = 'ibm_brisbane'  # change to a backend your account has access to
IBM_SHOTS = 1024              # fewer shots = faster + cheaper, noisier
N_DEMO_SAMPLES = 3            # number of test images to run on hardware

model_qpu = QMedViT_Softmax_Only(
    stem_chs=[64, 32, 64], depths=[3, 4, 10, 3], path_dropout=0.1,
    num_classes=n_classes,
    qpu_mode=True,
    qpu_shots=IBM_SHOTS,
    qdevice='qiskit.remote',
    qbackend=IBM_BACKEND,
    quantum_block_indices=(2,),  # only the last LTB of stage 3 -> 3x fewer calls
)
model_qpu.load_state_dict(model.state_dict(), strict=False)
model_qpu.eval()
# Run on CPU: the quantum part dispatches to the remote backend; only
# the classical layers run locally and the model is small enough.
print(f'Hardware model ready: backend={IBM_BACKEND}, shots={IBM_SHOTS}')

In [ ]:
import time
from torch.utils.data import Subset
from qiskit_ibm_runtime import Session

demo_loader = data.DataLoader(
    Subset(test_dataset, list(range(N_DEMO_SAMPLES))),
    batch_size=1, shuffle=False,
)

# A Session keeps the backend reserved for the duration of the block, so
# consecutive jobs incur seconds of queue, not hours. Without it each
# forward could wait hours in the public queue.
preds, labels_all, times = [], [], []
with Session(service=service, backend=IBM_BACKEND) as session:
    print(f'Opened IBM Runtime Session on {IBM_BACKEND} (id={session.session_id})')
    with torch.no_grad():
        for i, (img, label) in enumerate(demo_loader):
            t0 = time.time()
            logits = model_qpu(img)               # forward dispatches to IBM
            elapsed = time.time() - t0
            pred = logits.argmax(dim=-1).item()
            true = int(label.item())
            preds.append(pred); labels_all.append(true); times.append(elapsed)
            print(f'[{i+1}/{N_DEMO_SAMPLES}] pred={pred} true={true} '
                  f'time={elapsed/60:.1f} min')

acc = sum(p == t for p, t in zip(preds, labels_all)) / len(preds)
print(f'\nHardware acc on {len(preds)} samples: {acc:.3f}')
print(f'Average time per sample: {sum(times)/len(times)/60:.1f} min')